# 模块 2：航迹关联与卡尔曼滤波示例

本 notebook 直接复用 `tracker.py` 中的纯算法，展示恒速目标的稳定 ID、`age`、预测/更新和空量测预测。不导入 ROS，不复制算法实现。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'code' / 'tracker.py').is_file():
    root = root.parent
assert (root / 'code' / 'tracker.py').is_file(), '未找到项目根目录'
sys.path.insert(0, str(root / 'code'))

import numpy as np
from tracker import KalmanTracker, track_state_to_track

print('tracker module:', root / 'code' / 'tracker.py')

## 1. 恒速量测：ID 稳定且速度收敛

以 10 Hz 提供一条三维恒速轨迹，每帧位移远小于 2 m 关联门限。

In [ ]:
tracker = KalmanTracker(process_noise=0.05, measurement_noise=0.05)
dt = 0.1
velocity = np.array([0.4, -0.2, 0.1])
rows = []
for frame in range(20):
    measurement = np.array([1.0, 2.0, 3.0]) + velocity * frame * dt
    state = tracker.step([measurement], dt=dt, timestamp=frame * dt)[0]
    if frame in (0, 1, 5, 10, 19):
        rows.append((frame, state.id, state.age, state.state[:3].copy(), state.state[3:].copy()))

print('%5s %3s %4s %-28s %s' % ('frame', 'id', 'age', 'position', 'velocity'))
for frame, track_id, age, position, estimate in rows:
    print('%5d %3d %4d %-28s %s' % (frame, track_id, age, np.round(position, 4), np.round(estimate, 4)))
assert {row[1] for row in rows} == {0}
assert rows[-1][2] == 20

## 2. 预测与更新

先单独预测一个时间步，再用新量测更新，观察位置修正。

In [ ]:
before = tracker.active_tracks[0].state.copy()
tracker.predict(0.2)
predicted = tracker.active_tracks[0].state.copy()
measurement = predicted[:3] + np.array([0.10, -0.05, 0.02])
association = tracker.associate([measurement])
tracker.update(measurement)
updated = tracker.active_tracks[0].state.copy()
print('before   :', np.round(before, 4))
print('predicted:', np.round(predicted, 4))
print('associate:', association)
print('updated  :', np.round(updated, 4))
assert association == {0: 0}

## 3. 短段空量测与恢复

空量测仍执行预测并增加 `age`；3 帧少于默认删除条件，恢复时应关联回原 ID。

In [ ]:
track_id = tracker.active_tracks[0].id
empty_rows = []
for missed in range(1, 4):
    state = tracker.step([], dt=dt, timestamp=None)[0]
    empty_rows.append((missed, state.id, state.age, state.state[:3].copy()))
for row in empty_rows:
    print('empty=%d id=%d age=%d predicted=%s' % (row[0], row[1], row[2], np.round(row[3], 4)))
recovery_measurement = tracker.active_tracks[0].state[:3] + np.array([0.02, 0.0, 0.0])
recovered = tracker.step([recovery_measurement], dt=dt, timestamp=3.0)[0]
published = track_state_to_track(recovered)
print('recovered:', published)
assert recovered.id == track_id